# Byte Pair Encoding (BPE) — Подробный разбор алгоритма

BPE — это популярный алгоритм для разбиения текста на субсловные юниты (токены), широко используемый в задачах NLP (Natural Language Processing), например, при обучении трансформеров.


## Принцип работы BPE

**Кратко:** BPE итерирует по исходному тексту, объединяя самые частотные пары символов в новые токены. Алгоритм продолжается до достижения желаемого размера словаря или заданного числа слияний. В результате большие слова разбиваются на устойчивые подстроки.

**Более подробно:**
1. **Токенизация**: разбиваем каждое слово корпуса на отдельные символы.
2. **Подсчёт пар**: находим частоты соседних пар символов по всему корпусу.
3. **Слияние самой частой пары**: объединяем самую частую пару в новый токен.
4. **Повторяем 2–3** до достижения нужного размера словаря или когда частоты новых пар становятся малыми.


In [70]:
corpus = [
    "This is the Hugging Face Course.",
    "This chapter is about tokenization.",
    "This section shows several tokenizer algorithms.",
    "Hopefully, you will be able to understand how they are trained and generate tokens.",
]

print("Корпус текстов:")
for i, text in enumerate(corpus, 1):
    print(f"{i}. {text}")

Корпус текстов:
1. This is the Hugging Face Course.
2. This chapter is about tokenization.
3. This section shows several tokenizer algorithms.
4. Hopefully, you will be able to understand how they are trained and generate tokens.


In [71]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("gpt2")
print("Тип pre-tokenizer'а:", tokenizer.backend_tokenizer.pre_tokenizer)

Тип pre-tokenizer'а: ByteLevel(add_prefix_space=False, trim_offsets=True, use_regex=True)


**GPT использует pre-tokenizer, который:**
- Разбивает текст на слова
- Сохраняет пробелы как специальный символ Ġ (это U+0120)
- Отделяет пунктуацию

In [ ]:
from collections import defaultdict

word_freqs = defaultdict(int)

for text in corpus:
    words_with_offsets = tokenizer.backend_tokenizer.pre_tokenizer.pre_tokenize_str(text)
    new_words = [word for word, offset in words_with_offsets]
    for word in new_words:
        word_freqs[word] += 1

print("Слова и их частоты:")
for word, freq in sorted(word_freqs.items(), key=lambda x: x[1], reverse=True):
    print(f"  '{word}': {freq}")

Слова и их частоты:
  '.': 4
  'This': 3
  'Ġis': 2
  'Ġthe': 1
  'ĠHugging': 1
  'ĠFace': 1
  'ĠCourse': 1
  'Ġchapter': 1
  'Ġabout': 1
  'Ġtokenization': 1
  'Ġsection': 1
  'Ġshows': 1
  'Ġseveral': 1
  'Ġtokenizer': 1
  'Ġalgorithms': 1
  'Hopefully': 1
  ',': 1
  'Ġyou': 1
  'Ġwill': 1
  'Ġbe': 1
  'Ġable': 1
  'Ġto': 1
  'Ġunderstand': 1
  'Ġhow': 1
  'Ġthey': 1
  'Ġare': 1
  'Ġtrained': 1
  'Ġand': 1
  'Ġgenerate': 1
  'Ġtokens': 1


Ġ — он обозначает пробел перед словом. Это позволяет модели отличать начало слова от середины.

**Создание начального словаря (базовые символы)**

Собираем все уникальные символы, которые встречаются в словах.

In [73]:
alphabet = set()
for word in word_freqs.keys():
    for char in word:
        alphabet.add(char)

alphabet = sorted(list(alphabet))
print(f"Алфавит корпуса ({len(alphabet)} символов):")
print(''.join(alphabet))

Алфавит корпуса (30 символов):
,.CFHTabcdefghiklmnoprstuvwyzĠ


In [74]:
# Добавляем специальный токен GPT-2 (обозначает конец текста)
vocab = ["<|endoftext|>"] + alphabet.copy()
print(f"Начальный словарь: {len(vocab)} токенов")
print(vocab) 

Начальный словарь: 31 токенов
['<|endoftext|>', ',', '.', 'C', 'F', 'H', 'T', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'k', 'l', 'm', 'n', 'o', 'p', 'r', 's', 't', 'u', 'v', 'w', 'y', 'z', 'Ġ']


Перед началом обучения каждое слово представляется как список составляющих его символов

In [75]:
splits = {word: [c for c in word] for word in word_freqs.keys()}
print("Пример разбиения слова 'This':", splits["This"])
print("Пример разбиения слова 'Ġtokenization':", splits["Ġtokenization"])

Пример разбиения слова 'This': ['T', 'h', 'i', 's']
Пример разбиения слова 'Ġtokenization': ['Ġ', 't', 'o', 'k', 'e', 'n', 'i', 'z', 'a', 't', 'i', 'o', 'n']


**Функция для подсчета частот пар**

На каждом шаге обучения мы считаем, сколько раз каждая пара соседних токенов встречается во всех словах

In [76]:
def compute_pair_freqs(splits, word_freqs):
    """Подсчет частоты всех пар токенов в текущем разбиении."""
    pair_freqs = defaultdict(int)
    for word, freq in word_freqs.items():
        split = splits[word]
        if len(split) < 2:
            continue
        for i in range(len(split) - 1):
            pair = (split[i], split[i+1])
            pair_freqs[pair] += freq
    return pair_freqs

initial_pairs = compute_pair_freqs(splits, word_freqs)
print("Первые 5 самых частых пар (начальное состояние):")
sorted_pairs = sorted(initial_pairs.items(), key=lambda x: x[1], reverse=True)
for pair, freq in sorted_pairs[:5]:
    print(f"  {pair}: {freq}")

Первые 5 самых частых пар (начальное состояние):
  ('Ġ', 't'): 7
  ('i', 's'): 5
  ('e', 'r'): 5
  ('Ġ', 'a'): 5
  ('t', 'o'): 4


**Функция слияния пары**

Склеиваем выбранную пару во всех словах.

In [77]:
def merge_pair(pair, splits):
    """Объединяет все вхождения пары (a, b) в a+b во всех словах."""
    a, b = pair
    new_splits = {}
    for word, split in splits.items():
        if len(split) < 2:
            new_splits[word] = split
            continue
        i = 0
        new_split = []
        while i < len(split):
            if i < len(split) - 1 and split[i] == a and split[i+1] == b:
                new_split.append(a + b)
                i += 2
            else:
                new_split.append(split[i])
                i += 1
        new_splits[word] = new_split
    return new_splits

In [78]:
current_splits = splits.copy()
merges = {} 
target_vocab_size = 50

print(f"Исходный размер словаря: {len(vocab)}")
print("="*50)

iteration = 1
while len(vocab) < target_vocab_size:
    # 1. Подсчитываем частоты пар
    pair_freqs = compute_pair_freqs(current_splits, word_freqs)
    if not pair_freqs:
        break
    
    # 2. Находим самую частую пару
    best_pair = max(pair_freqs.items(), key=lambda x: x[1])[0]
    best_freq = pair_freqs[best_pair]
    
    # 3. Выполняем слияние
    current_splits = merge_pair(best_pair, current_splits)
    new_token = best_pair[0] + best_pair[1]
    merges[best_pair] = new_token
    vocab.append(new_token)
    
    print(f"Шаг {iteration:2d}: слияние {best_pair} -> '{new_token}' (частота: {best_freq})")
    iteration += 1

print("="*50)
print(f"Размер словаря: {len(vocab)}")


Исходный размер словаря: 31
Шаг  1: слияние ('Ġ', 't') -> 'Ġt' (частота: 7)
Шаг  2: слияние ('i', 's') -> 'is' (частота: 5)
Шаг  3: слияние ('e', 'r') -> 'er' (частота: 5)
Шаг  4: слияние ('Ġ', 'a') -> 'Ġa' (частота: 5)
Шаг  5: слияние ('Ġt', 'o') -> 'Ġto' (частота: 4)
Шаг  6: слияние ('e', 'n') -> 'en' (частота: 4)
Шаг  7: слияние ('T', 'h') -> 'Th' (частота: 3)
Шаг  8: слияние ('Th', 'is') -> 'This' (частота: 3)
Шаг  9: слияние ('o', 'u') -> 'ou' (частота: 3)
Шаг 10: слияние ('s', 'e') -> 'se' (частота: 3)
Шаг 11: слияние ('Ġto', 'k') -> 'Ġtok' (частота: 3)
Шаг 12: слияние ('Ġtok', 'en') -> 'Ġtoken' (частота: 3)
Шаг 13: слияние ('n', 'd') -> 'nd' (частота: 3)
Шаг 14: слияние ('Ġ', 'is') -> 'Ġis' (частота: 2)
Шаг 15: слияние ('Ġt', 'h') -> 'Ġth' (частота: 2)
Шаг 16: слияние ('Ġth', 'e') -> 'Ġthe' (частота: 2)
Шаг 17: слияние ('i', 'n') -> 'in' (частота: 2)
Шаг 18: слияние ('Ġa', 'b') -> 'Ġab' (частота: 2)
Шаг 19: слияние ('Ġtoken', 'i') -> 'Ġtokeni' (частота: 2)
Размер словаря: 50


**Анализ выученных слияний**

Посмотрим, какие подслова научилась склеивать модель

In [79]:
print("\nПервые 10 выученных слияний:")
for i, (pair, merged) in enumerate(list(merges.items())[:10], 1):
    print(f"{i:2d}. {pair} -> '{merged}'")


Первые 10 выученных слияний:
 1. ('Ġ', 't') -> 'Ġt'
 2. ('i', 's') -> 'is'
 3. ('e', 'r') -> 'er'
 4. ('Ġ', 'a') -> 'Ġa'
 5. ('Ġt', 'o') -> 'Ġto'
 6. ('e', 'n') -> 'en'
 7. ('T', 'h') -> 'Th'
 8. ('Th', 'is') -> 'This'
 9. ('o', 'u') -> 'ou'
10. ('s', 'e') -> 'se'


In [82]:
merges

{('Ġ', 't'): 'Ġt',
 ('i', 's'): 'is',
 ('e', 'r'): 'er',
 ('Ġ', 'a'): 'Ġa',
 ('Ġt', 'o'): 'Ġto',
 ('e', 'n'): 'en',
 ('T', 'h'): 'Th',
 ('Th', 'is'): 'This',
 ('o', 'u'): 'ou',
 ('s', 'e'): 'se',
 ('Ġto', 'k'): 'Ġtok',
 ('Ġtok', 'en'): 'Ġtoken',
 ('n', 'd'): 'nd',
 ('Ġ', 'is'): 'Ġis',
 ('Ġt', 'h'): 'Ġth',
 ('Ġth', 'e'): 'Ġthe',
 ('i', 'n'): 'in',
 ('Ġa', 'b'): 'Ġab',
 ('Ġtoken', 'i'): 'Ġtokeni'}

In [80]:
# Посмотрим, как теперь выглядят разбиения некоторых слов
print("\nПримеры разбиений слов после обучения:")
test_words = ["This", "Ġtokenization", "Ġtrained", "Ġalgorithms"]
for word in test_words:
    if word in current_splits:
        print(f"'{word}': {current_splits[word]}")


Примеры разбиений слов после обучения:
'This': ['This']
'Ġtokenization': ['Ġtokeni', 'z', 'a', 't', 'i', 'o', 'n']
'Ġtrained': ['Ġt', 'r', 'a', 'in', 'e', 'd']
'Ġalgorithms': ['Ġa', 'l', 'g', 'o', 'r', 'i', 't', 'h', 'm', 's']


In [ ]:
def bpe_tokenize_with_ids(text, vocab, merges, pre_tokenizer):
    """
    BPE токенизация: возвращает два списка — токены и их индексы в vocab.
    :param text: исходный текст
    :param vocab: список токенов (например, ['<|endoftext|>', ..., "Th", "is", ...])
    :param merges: OrderedDict пар (a, b) -> "ab", порядок важен!
    :param pre_tokenizer: pre_tokenizer от GPT2 (например, tokenizer.backend_tokenizer.pre_tokenizer)
    :return: (tokens, token_ids)
    """
    words_with_offsets = pre_tokenizer.pre_tokenize_str(text)
    words = [word for word, offset in words_with_offsets]
    tokens = []

    merges_keys = list(merges.keys())
    merges_set = set(merges_keys)

    for word in words:
        symbols = [c for c in word]
        while True:
            pairs = [(symbols[i], symbols[i + 1]) for i in range(len(symbols) - 1)]
            mergeable = [pair for pair in pairs if pair in merges_set]
            if not mergeable:
                break
            # Жадно: ищем первую по merges
            min_idx = min(merges_keys.index(pair) for pair in mergeable)
            to_merge = merges_keys[min_idx]
            # Сливаем все такие пары
            i = 0
            new_symbols = []
            while i < len(symbols):
                if i < len(symbols) - 1 and (symbols[i], symbols[i + 1]) == to_merge:
                    new_symbols.append(symbols[i] + symbols[i + 1])
                    i += 2
                else:
                    new_symbols.append(symbols[i])
                    i += 1
            symbols = new_symbols
        tokens.extend(symbols)

    # Получаем индексы токенов
    token_ids = [vocab.index(t) if t in vocab else None for t in tokens]
    return tokens, token_ids


tokens, ids = bpe_tokenize_with_ids(
    "This is tokenization.", vocab, merges, tokenizer.backend_tokenizer.pre_tokenizer
)
print(tokens)
print(ids)

['This', 'Ġis', 'Ġtokeni', 'z', 'a', 't', 'i', 'o', 'n', '.']
[38, 44, 49, 29, 7, 24, 15, 20, 19, 2]
